# Temperature Downscaling — Baselines

**Goal:** Evaluate baseline methods for downscaling MODIS Land Surface Temperature from 5km back to 1km.

**Pipeline:**
1. Load MODIS 1km LST as ground truth
2. Coarsen to ~5km using bicubic interpolation (simulate low-resolution input)
3. Upsample back to 1km using two baselines:
   - **BCSD** (Bias Correction Spatial Disaggregation) — classical statistical downscaling ([Wood et al., 2004](https://doi.org/10.1023/B:CLIM.0000013685.99609.9e))
   - **Lasso Regression** — L1-regularized linear model with spatial features
4. Compute metrics: RMSE, MAE, PSNR, SSIM

Reference: [Vandal et al. 2017 — DeepSD](https://arxiv.org/abs/1703.03126)

## 0. Setup

In [ ]:
!pip install -q rasterio pyhdf huggingface_hub scikit-learn scikit-image

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from collections import defaultdict

from scipy.ndimage import zoom
from sklearn.linear_model import Lasso
from skimage.metrics import structural_similarity as ssim

from pyhdf.SD import SD, SDC
from huggingface_hub import snapshot_download

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

SCALE_FACTOR = 5  # 1km * 5 = 5km

## 1. Download Data

In [ ]:
DATA_ROOT = 'data'

# Download metadata + splits + ASTER DEM
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
    ignore_patterns=['MODIS/*', 'Sentinel2/*', 'LULC/*'],
)

# Download MODIS for full years 2022 (train/val) and 2023 (test)
modis_patterns = (
    [f'MODIS/*.A2022{d:03d}.*' for d in range(1, 366)] +
    [f'MODIS/*.A2023{d:03d}.*' for d in range(1, 366)]
)
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
    allow_patterns=modis_patterns,
)

MODIS_DIR = os.path.join(DATA_ROOT, 'MODIS')
ASTER_DIR = os.path.join(DATA_ROOT, 'ASTER')
print(f"MODIS files: {len(glob.glob(os.path.join(MODIS_DIR, '*.hdf')))}")
print(f"ASTER files: {len(glob.glob(os.path.join(ASTER_DIR, '*_dem.tif')))}")

## 2. Load MODIS LST Data

Read each MODIS tile's LST, convert to Celsius, and collect individual 1200x1200 tiles as ground-truth samples. Each tile is one sample — we work per-tile to avoid mosaicing complexity and keep things Colab-friendly.

In [ ]:
def read_modis_lst(hdf_path, layer='LST_Day_1km'):
    """Read MODIS LST from HDF and convert to Celsius."""
    hdf = SD(hdf_path, SDC.READ)
    sds = hdf.select(layer)
    data = sds.get().astype(np.float64)
    attrs = sds.attributes()
    scale = attrs.get('scale_factor', 0.02)
    valid_range = attrs.get('valid_range', [7500, 65535])
    data = np.where(
        (data >= valid_range[0]) & (data <= valid_range[1]),
        data * scale - 273.15,
        np.nan,
    )
    hdf.end()
    return data


def parse_modis_filename(path):
    """Extract date and tile from MODIS filename."""
    fname = os.path.basename(path)
    m = re.match(r'MOD11A1\.A(\d{7})\.(h\d{2}v\d{2})\.', fname)
    if not m:
        return None, None
    doy_str, tile = m.groups()
    year = int(doy_str[:4])
    doy = int(doy_str[4:])
    date = datetime(year, 1, 1) + timedelta(days=doy - 1)
    return date, tile

In [ ]:
modis_hdf = sorted(glob.glob(os.path.join(MODIS_DIR, 'MOD11A1.*.hdf')))
print(f"Total MODIS HDF files: {len(modis_hdf)}")

# Stream files: extract patches immediately and discard the full 1200x1200 LST.
# Only small float32 patches are kept in memory.
MIN_VALID_FRAC = 0.5
PATCH_SIZE_PRE = 60   # must match PATCH_SIZE in next section
MAX_NAN_FRAC_PRE = 0.2

def _extract_patches_stream(lst, patch_size=PATCH_SIZE_PRE, max_nan_frac=MAX_NAN_FRAC_PRE):
    h, w = lst.shape
    out = []
    for i in range(0, h - patch_size + 1, patch_size):
        for j in range(0, w - patch_size + 1, patch_size):
            patch = lst[i:i+patch_size, j:j+patch_size]
            nan_frac = np.sum(np.isnan(patch)) / patch.size
            if nan_frac <= max_nan_frac:
                patch_filled = patch.astype(np.float32, copy=True)
                mask = np.isnan(patch_filled)
                if mask.any():
                    patch_filled[mask] = np.nanmean(patch_filled)
                out.append(patch_filled)
    return out

patches_2022_by_month = defaultdict(list)  # month -> list of float32 patches
patches_2023 = []
n_kept_files = 0
tiles_seen = set()
dates_seen = []

for f in modis_hdf:
    date, tile = parse_modis_filename(f)
    if date is None:
        continue
    lst = read_modis_lst(f)
    valid_frac = np.sum(~np.isnan(lst)) / lst.size
    if valid_frac < MIN_VALID_FRAC:
        del lst
        continue
    patches = _extract_patches_stream(lst)
    del lst  # free the 1200x1200 array immediately
    if not patches:
        continue
    if date.year == 2022:
        patches_2022_by_month[date.month].extend(patches)
    elif date.year == 2023:
        patches_2023.extend(patches)
    else:
        continue
    n_kept_files += 1
    tiles_seen.add(tile)
    dates_seen.append(date)

n_2022 = sum(len(v) for v in patches_2022_by_month.values())
print(f"Files kept: {n_kept_files} (>= {MIN_VALID_FRAC:.0%} valid)")
print(f"Patches — 2022: {n_2022}, 2023: {len(patches_2023)}")
print(f"Tiles: {sorted(tiles_seen)}")
if dates_seen:
    print(f"Date range: {min(dates_seen):%Y-%m-%d} to {max(dates_seen):%Y-%m-%d}")

## 3. Extract Patches & Build Train/Val/Test Splits

Extract non-overlapping 60x60 patches (12x12 at 5km LR). Splits:

- **Train + Val**: year 2022. For each calendar month, 80% of patches go to train and 20% to val (stratified per month so all seasons are represented in both).
- **Test**: year 2023.

In [ ]:
PATCH_SIZE = PATCH_SIZE_PRE          # 60
LR_PATCH = PATCH_SIZE // SCALE_FACTOR  # 12

# Patches were already extracted (streaming) in the previous cell into
# `patches_2022_by_month` and `patches_2023`. Here we just build the splits.

np.random.seed(42)
train_list, val_list = [], []
for month in sorted(patches_2022_by_month.keys()):
    month_patches = np.stack(patches_2022_by_month[month]).astype(np.float32)
    patches_2022_by_month[month] = None  # free
    idx = np.random.permutation(len(month_patches))
    cut = int(0.8 * len(month_patches))
    train_list.append(month_patches[idx[:cut]])
    val_list.append(month_patches[idx[cut:]])
    print(f"  Month {month:02d}: {len(month_patches)} patches "
          f"-> {cut} train / {len(month_patches) - cut} val")

train_hr = np.concatenate(train_list, axis=0).astype(np.float32)
val_hr   = np.concatenate(val_list,   axis=0).astype(np.float32)
test_hr  = np.stack(patches_2023).astype(np.float32) if patches_2023 else np.empty((0, PATCH_SIZE, PATCH_SIZE), np.float32)

# Free intermediate references
del train_list, val_list, patches_2022_by_month, patches_2023
import gc; gc.collect()

print(f"\nTrain (2022): {len(train_hr)}")
print(f"Val   (2022): {len(val_hr)}")
print(f"Test  (2023): {len(test_hr)}")

## 4. Coarsen HR → LR (Bicubic) & Prepare Inputs

Simulate low-resolution input by downsampling each 60x60 HR patch to 12x12 using bicubic interpolation, then upsample back to 60x60 (also bicubic) as the input for all methods.

In [ ]:
def coarsen_bicubic(hr, scale=SCALE_FACTOR):
    """Downsample HR to LR using bicubic interpolation."""
    return zoom(hr, 1.0 / scale, order=3)

def upsample_bicubic(lr, scale=SCALE_FACTOR):
    """Upsample LR to HR using bicubic interpolation."""
    return zoom(lr, scale, order=3)


# Only precompute the small LR arrays (12x12). The large bicubic-upsampled
# HR (60x60) copies are generated on-the-fly per batch in the BCSD/Lasso
# cells to keep peak RAM low.
train_lr = np.stack([coarsen_bicubic(p) for p in train_hr]).astype(np.float32)
val_lr   = np.stack([coarsen_bicubic(p) for p in val_hr]).astype(np.float32)
test_lr  = np.stack([coarsen_bicubic(p) for p in test_hr]).astype(np.float32)

print(f"HR: {train_hr.shape[1:]}, LR: {train_lr.shape[1:]}  (bicubic computed lazily)")

## 5. Metrics

Define evaluation metrics: RMSE, MAE, PSNR, and SSIM.

In [ ]:
def compute_metrics(pred, gt):
    """Compute RMSE, MAE, PSNR, and SSIM between prediction and ground truth."""
    mse = np.mean((pred - gt) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(pred - gt))
    # PSNR: use the data range of ground truth
    data_range = gt.max() - gt.min()
    if data_range == 0 or mse == 0:
        psnr = float('inf')
    else:
        psnr = 10 * np.log10(data_range ** 2 / mse)
    ssim_val = ssim(gt, pred, data_range=data_range)
    return {'RMSE': rmse, 'MAE': mae, 'PSNR': psnr, 'SSIM': ssim_val}


def evaluate_method(preds, gts):
    """Compute mean metrics across all test patches."""
    all_metrics = defaultdict(list)
    for pred, gt in zip(preds, gts):
        m = compute_metrics(pred, gt)
        for k, v in m.items():
            all_metrics[k].append(v)
    return {k: np.mean(v) for k, v in all_metrics.items()}

## 6. Baseline 1 — BCSD (Bias Correction Spatial Disaggregation)

BCSD ([Wood et al., 2004](https://doi.org/10.1023/B:CLIM.0000013685.99609.9e)) is a classical statistical downscaling method. For temperature (an additive variable):

1. Compute HR climatology (mean over training set at 1km)
2. Compute LR climatology (mean over training set at 5km), upsample to 1km
3. Spatial detail pattern = HR climatology − upsampled LR climatology
4. For each test sample: prediction = bicubic-upsampled LR + spatial detail pattern

This transfers fine-scale spatial structure from the climatology to each daily prediction.

In [ ]:
# Compute climatologies from training data only
hr_climatology = np.mean(train_hr, axis=0).astype(np.float32)            # (60, 60)
lr_climatology = np.mean(train_lr, axis=0).astype(np.float32)            # (12, 12)
lr_clim_upsampled = upsample_bicubic(lr_climatology).astype(np.float32)  # (60, 60)

# Spatial detail pattern (additive for temperature)
spatial_detail = (hr_climatology - lr_clim_upsampled).astype(np.float32)

print(f"HR climatology range: [{hr_climatology.min():.1f}, {hr_climatology.max():.1f}] °C")
print(f"Spatial detail range: [{spatial_detail.min():.2f}, {spatial_detail.max():.2f}] °C")

def bcsd_predict(lr_arr):
    """BCSD prediction streamed per patch — no full bicubic array in memory."""
    out = np.empty((len(lr_arr), PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
    for i, lr in enumerate(lr_arr):
        out[i] = upsample_bicubic(lr).astype(np.float32) + spatial_detail
    return out

val_bcsd  = bcsd_predict(val_lr)
test_bcsd = bcsd_predict(test_lr)

bcsd_val_metrics  = evaluate_method(val_bcsd,  val_hr)
bcsd_test_metrics = evaluate_method(test_bcsd, test_hr)
print(f"\nBCSD val  (2022): {', '.join(f'{k}: {v:.4f}' for k, v in bcsd_val_metrics.items())}")
print(f"BCSD test (2023): {', '.join(f'{k}: {v:.4f}' for k, v in bcsd_test_metrics.items())}")

## 7. Baseline 2 — Lasso Regression

Per-pixel Lasso regression following [Vandal et al., 2017]. For each HR pixel, we build features from the bicubic-upsampled LR image:
- The pixel value itself
- A local 3x3 neighborhood (spatial context)
- Normalized (row, col) coordinates (positional encoding)

The Lasso (L1-regularized linear model) learns a mapping from these features to the true HR pixel value.

In [ ]:
def build_lasso_features(bicubic_patches):
    """Build per-pixel feature matrix from bicubic-upsampled patches.

    Features per pixel: 3x3 neighborhood (9) + normalized row/col (2) = 11.
    Vectorized + float32 to keep peak memory low.
    """
    bicubic_patches = np.ascontiguousarray(bicubic_patches, dtype=np.float32)
    n, h, w = bicubic_patches.shape
    padded = np.pad(bicubic_patches, ((0, 0), (1, 1), (1, 1)), mode='reflect')

    feats = np.empty((n, h, w, 11), dtype=np.float32)
    k = 0
    for di in range(3):
        for dj in range(3):
            feats[..., k] = padded[:, di:di + h, dj:dj + w]
            k += 1
    rows_norm = np.linspace(0, 1, h, dtype=np.float32)
    cols_norm = np.linspace(0, 1, w, dtype=np.float32)
    rr, cc = np.meshgrid(rows_norm, cols_norm, indexing='ij')
    feats[..., 9]  = rr
    feats[..., 10] = cc
    return feats.reshape(n * h * w, 11)


print("Building Lasso features (this may take a minute)...")

# Subsample training data and upsample only that subset on the fly.
max_train = min(len(train_lr), 500)
train_bicubic_sub = np.stack(
    [upsample_bicubic(lr) for lr in train_lr[:max_train]]
).astype(np.float32)
X_train = build_lasso_features(train_bicubic_sub)
y_train = train_hr[:max_train].ravel().astype(np.float32)
del train_bicubic_sub
import gc; gc.collect()

print(f"Training features: {X_train.shape}, targets: {y_train.shape}")

In [ ]:
print("Training Lasso model...")
lasso = Lasso(alpha=0.01, max_iter=5000, warm_start=True)
lasso.fit(X_train, y_train)
print(f"Lasso coefficients: {lasso.coef_}")
print(f"Lasso intercept: {lasso.intercept_:.4f}")
print(f"Non-zero coefficients: {np.sum(lasso.coef_ != 0)}/{len(lasso.coef_)}")

# Free training feature matrix before building val/test features
del X_train, y_train
import gc; gc.collect()

def predict_in_batches(lr_arr, batch=32):
    """For each batch: bicubic upsample LR -> features -> predict.
    Nothing full-sized lives outside the loop."""
    n = len(lr_arr)
    out = np.empty((n, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
    for start in range(0, n, batch):
        end = min(start + batch, n)
        bic = np.stack([upsample_bicubic(lr) for lr in lr_arr[start:end]]).astype(np.float32)
        X = build_lasso_features(bic)
        del bic
        y = lasso.predict(X).astype(np.float32)
        del X
        out[start:end] = y.reshape(end - start, PATCH_SIZE, PATCH_SIZE)
        del y
    return out

print("\nPredicting on val set...")
val_lasso = predict_in_batches(val_lr)
gc.collect()

print("Predicting on test set...")
test_lasso = predict_in_batches(test_lr)
gc.collect()

lasso_val_metrics  = evaluate_method(val_lasso,  val_hr)
lasso_test_metrics = evaluate_method(test_lasso, test_hr)
print(f"\nLasso val  (2022): {', '.join(f'{k}: {v:.4f}' for k, v in lasso_val_metrics.items())}")
print(f"Lasso test (2023): {', '.join(f'{k}: {v:.4f}' for k, v in lasso_test_metrics.items())}")

## 8. Results Comparison

In [ ]:
results_val = {
    'BCSD':  bcsd_val_metrics,
    'Lasso': lasso_val_metrics,
}
results_test = {
    'BCSD':  bcsd_test_metrics,
    'Lasso': lasso_test_metrics,
}

# Backwards compat for downstream cells
results = results_test

def _print_table(title, res):
    print(title)
    print(f"{'Method':<10} {'RMSE':>8} {'MAE':>8} {'PSNR':>8} {'SSIM':>8}")
    print("-" * 46)
    for method, metrics in res.items():
        print(f"{method:<10} {metrics['RMSE']:>8.4f} {metrics['MAE']:>8.4f} "
              f"{metrics['PSNR']:>8.2f} {metrics['SSIM']:>8.4f}")
    print()

_print_table("Validation (2022)", results_val)
_print_table("Test (2023)",       results_test)

In [ ]:
# Bar chart comparison
metrics_names = ['RMSE', 'MAE', 'PSNR', 'SSIM']
methods = list(results.keys())
colors = ['#2196F3', '#FF9800']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric in zip(axes, metrics_names):
    vals = [results[m][metric] for m in methods]
    bars = ax.bar(methods, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Baseline Comparison: BCSD vs Lasso (5km → 1km)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Visual Comparison

Side-by-side comparison of a few test patches: ground truth HR, bicubic-upsampled LR input, and both baseline predictions.

In [ ]:
n_show = 4
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
col_titles = ['Ground Truth (1km)', 'Bicubic Input (5km→1km)', 'BCSD', 'Lasso']

# Pick evenly spaced test samples
show_idx = np.linspace(0, len(test_hr) - 1, n_show, dtype=int)

for row, idx in enumerate(show_idx):
    bicubic_img = upsample_bicubic(test_lr[idx]).astype(np.float32)  # on the fly
    images = [test_hr[idx], bicubic_img, test_bcsd[idx], test_lasso[idx]]
    vmin, vmax = test_hr[idx].min(), test_hr[idx].max()

    for col, (img, title) in enumerate(zip(images, col_titles)):
        ax = axes[row, col]
        im = ax.imshow(img, cmap='RdYlBu_r', vmin=vmin, vmax=vmax)
        if row == 0:
            ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])

        # Show per-patch RMSE for predictions
        if col >= 2:
            rmse = np.sqrt(np.mean((img - test_hr[idx]) ** 2))
            ax.text(1, 1, f'RMSE: {rmse:.2f}', transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=9, color='white',
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))

    fig.colorbar(im, ax=axes[row, :].tolist(), shrink=0.8, label='LST (°C)')

plt.suptitle('Visual Comparison: Test Patches', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10. Error Distribution

In [ ]:
# Per-pixel error distributions
bcsd_errors = (test_bcsd - test_hr).ravel()
lasso_errors = (test_lasso - test_hr).ravel()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histograms
bins = np.linspace(-5, 5, 100)
ax1.hist(bcsd_errors, bins=bins, alpha=0.6, label='BCSD', color='#2196F3', density=True)
ax1.hist(lasso_errors, bins=bins, alpha=0.6, label='Lasso', color='#FF9800', density=True)
ax1.axvline(0, color='black', linestyle='--', linewidth=0.8)
ax1.set_xlabel('Prediction Error (°C)')
ax1.set_ylabel('Density')
ax1.set_title('Error Distribution (Predicted − Ground Truth)')
ax1.legend()
ax1.grid(alpha=0.3)

# Per-patch RMSE distribution
bcsd_patch_rmse = [np.sqrt(np.mean((p - g) ** 2)) for p, g in zip(test_bcsd, test_hr)]
lasso_patch_rmse = [np.sqrt(np.mean((p - g) ** 2)) for p, g in zip(test_lasso, test_hr)]

bp = ax2.boxplot([bcsd_patch_rmse, lasso_patch_rmse], labels=['BCSD', 'Lasso'],
                  patch_artist=True)
bp['boxes'][0].set_facecolor('#2196F3')
bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#FF9800')
bp['boxes'][1].set_alpha(0.6)
ax2.set_ylabel('RMSE (°C)')
ax2.set_title('Per-Patch RMSE Distribution')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"BCSD  — median patch RMSE: {np.median(bcsd_patch_rmse):.3f} °C")
print(f"Lasso — median patch RMSE: {np.median(lasso_patch_rmse):.3f} °C")

## 11. Going Further: 1km → ~200m

The baselines above are trained for a 5x scale jump (5km → 1km). We can apply the *same* trained models in cascade to push beyond MODIS's native resolution: treat each 1km test patch as the "low-res" input and produce a synthetic ~200m output (another 5x).

There is no ground truth at 200m, so this section is **visualization-only** — no metrics. The goal is to inspect how each baseline extrapolates fine-scale structure beyond the data it was trained on.

In [ ]:
# Pick a few test patches at 1km, treat as LR, upscale 5x to ~200m.
n_show2 = 4
show_idx2 = np.linspace(0, len(test_hr) - 1, n_show2, dtype=int)

# 1km input -> ~200m bicubic upsampled (the input space for both baselines)
finer_inputs   = np.array([upsample_bicubic(test_hr[i]) for i in show_idx2])  # (n, 300, 300)

# --- BCSD at the new scale ---
# Need a "spatial detail" pattern at 300x300 from a 60x60 climatology.
# Use the 60x60 train HR climatology as the "LR" climatology at this finer step,
# upsample it 5x and subtract to get an additive detail field of zeros — i.e.
# without finer-scale ground truth, BCSD has no extra structure to add and
# collapses to bicubic. Make this explicit by reusing the bicubic input.
finer_bcsd = finer_inputs.copy()

# --- Lasso at the new scale ---
# The trained model is purely local (3x3 neighborhood + normalized coords),
# so it generalizes to any spatial size. Apply it directly.
X_finer = build_lasso_features(finer_inputs)
finer_lasso = lasso.predict(X_finer).reshape(finer_inputs.shape)

print(f"1km input shape:  {test_hr[show_idx2[0]].shape}")
print(f"~200m output shape: {finer_inputs.shape[1:]}")

In [ ]:
fig, axes = plt.subplots(n_show2, 3, figsize=(12, 4 * n_show2))
col_titles = ['Input (1km)', 'BCSD → ~200m', 'Lasso → ~200m']

for row, idx in enumerate(show_idx2):
    images = [test_hr[idx], finer_bcsd[row], finer_lasso[row]]
    vmin, vmax = test_hr[idx].min(), test_hr[idx].max()
    for col, (img, title) in enumerate(zip(images, col_titles)):
        ax = axes[row, col]
        im = ax.imshow(img, cmap='RdYlBu_r', vmin=vmin, vmax=vmax)
        if row == 0:
            ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(im, ax=axes[row, :].tolist(), shrink=0.8, label='LST (°C)')

plt.suptitle('1km → ~200m (no ground truth — visualization only)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()